In [1]:
import MDAnalysis as mda
from monomer_classes import *
from tqdm import tqdm
import yaml

/Users/nate_schwindt/anaconda3/envs/mda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
with open('charges.yaml', 'r') as file: # read in charges from yaml file
        charges_dict = yaml.safe_load(file)

In [7]:
# Check yaml charges for each fragment type
for frag in charges_dict:
    charge = round(np.sum([charges_dict[frag][atom]['charge'] for atom in charges_dict[frag]]),8)
    print(frag, charge)

MPD_L -0.15680002
MPD_L_Cl -0.19916003
MPD_T -0.07696003
TMC_L_P_1 0.15392007
TMC_L_P_2 0.15392007
TMC_L_P_3 0.15392007
TMC_L_D_1 -0.74802001
TMC_L_D_2 -0.74802001
TMC_L_D_3 -0.74802001
TMC_T_2P_1 0.07840002
TMC_T_2P_2 0.07840002
TMC_T_2P_3 0.07840002
TMC_T_1P_1_2 -0.87691007
TMC_T_1P_1_3 -0.87691007
TMC_T_1P_2_1 -0.87691007
TMC_T_1P_2_3 -0.87691007
TMC_T_1P_3_1 -0.87691007
TMC_T_1P_3_2 -0.87691007
TMC_T_0P_1 -1.85471996
TMC_T_0P_2 -1.85471996
TMC_T_0P_3 -1.85471996
TMC_C 0.19542


In [8]:
u = mda.Universe('chlorinated.tpr', 'chlorinated.gro')
atoms = u.atoms
print(f'Charge before reassignment: {atoms.charges.sum()}')

Charge before reassignment: 1.9096594769507647e-05


In [ ]:
for N in atoms.select_atoms('type n'):
    has_cl = len([dih for dih in N.dihedrals if 'cl' in dih.atoms.types]) > 0
    if not has_cl:
        mpd = MPD(N, xlink_n='n', ar_c='ca', term_n='nh', ar_h='ha', hn='hn')
        mpd.assign_charges(charges_dict['MPD_L'])
    else:
        mpd_cl = MPD_Cl(N, cl='cl', xlink_n='n', ar_c='ca', term_n='nh', ar_h='ha', hn='hn')
        mpd_cl.assign_charges(charges_dict['MPD_L_Cl'])

print('Original MPD-L:')
for atom in mpd.atoms:
    print(atom.type, atom.charge)

print('\nChlorinated MPD-L:')
for atom in mpd_cl.atoms:
    print(atom.type, atom.charge)

Original MPD-L:
n -0.46721
ca 0.09408
ca -0.08205001
ca -0.17455
ca 0.09408
ca -0.23816
ca -0.17455
nh -0.46721
ha 0.14181
ha 0.17558999
ha 0.17558999
ha 0.12763999
hn 0.31907001
hn 0.31907001
hn 0.39148

Chlorinated MPD-L:
n -0.46721
ca 0.10823
ca -0.08205001
ca -0.17455
ca 0.10823
ca -0.09627
ca -0.17455
n -0.46721
ha 0.14181
ha 0.17558999
ha 0.17558999
cl -0.10763
hn 0.33043
hn 0.33043


In [32]:
changed_atoms = ['ca3', 'ca6', 'ca5', 'cl', 'hn1a', 'hn2a']

charge_diff = np.zeros(len(changed_atoms))
for a,atom in enumerate(changed_atoms):
    if atom == 'cl':
        a1 = 'ha6'
        a2 = 'cl'
    else:
        a1 = atom
        a2 = atom

    charge_diff[a] = getattr(mpd_cl, a2).charge - getattr(mpd, a1).charge
    print(atom, charge_diff[a])

print('\nTotal changed charge:', charge_diff.sum())

ca3 0.01415000000000001
ca6 0.14189000000000002
ca5 0.01415000000000001
cl -0.23526999
hn1a 0.011359989999999986
hn2a 0.011359989999999986

Total changed charge: -0.04236001000000003


In [38]:
unchanged_atoms = ['ca1', 'ca2', 'ca4', 'ha1', 'ha2', 'ha4', 'N1', 'N2']

charge_to_distribute = -charge_diff.sum()
original_charges = np.zeros(len(unchanged_atoms))
for i, atom in enumerate(unchanged_atoms):
    original_charges[i] = getattr(mpd, atom).charge

charges_per_atom = charge_to_distribute * (original_charges / original_charges.sum())

for i, atom in enumerate(unchanged_atoms):
    old_charge = getattr(mpd, atom).charge
    new_charge = old_charge + charges_per_atom[i]
    print(f'{atom} {old_charge} -> {new_charge:.8f}')

ca1 -0.08205001 -> -0.07806683
ca2 -0.17455 -> -0.16607635
ca4 -0.17455 -> -0.16607635
ha1 0.14181 -> 0.13492573
ha2 0.17558999 -> 0.16706585
ha4 0.17558999 -> 0.16706585
N1 -0.46721 -> -0.44452896
N2 -0.46721 -> -0.44452896


In [39]:
with open('charges_modified.yaml', 'r') as file: # read in charges from yaml file
        charges_modified_dict = yaml.safe_load(file)

In [40]:
print('Creating MPD monomer fragments...')
mpds = []
Ns = []
for N in tqdm(atoms.select_atoms('type n nh')):
    has_cl = len([dih for dih in N.dihedrals if 'cl' in dih.atoms.types]) > 0

    if has_cl:
        mpd = MPD_Cl(N, cl='cl', xlink_n='n', ar_c='ca', term_n='nh', ar_h='ha', hn='hn')
    else:
        mpd = MPD(N, xlink_n='n', ar_c='ca', term_n='nh', ar_h='ha', hn='hn')
        
    if mpd.N2 not in Ns:
        mpd.assign_charges(charges_modified_dict[mpd.name])
        mpds.append(mpd)
    
    Ns.append(mpd.N1)
    Ns.append(mpd.N2)

print('Creating TMC monomer fragments...')
tmcs = []
Cs = []
for C in tqdm(atoms.select_atoms('type c')):
    tmc = TMC(C, ar_c='ca', xlink_c='c', ar_h='ha', deprot_o='o', prot_o='oh', ho_type='ho')
    if tmc.C2 not in Cs and tmc.C3 not in Cs:
        tmc.assign_charges(charges_modified_dict[tmc.name])
        tmcs.append(tmc)

    Cs.append(tmc.C1)
    Cs.append(tmc.C2)
    Cs.append(tmc.C3)


n_mpd_mono = len([mono for mono in mpds if mono.name == 'MPD'])
n_mpd_L = len([mono for mono in mpds if mono.name == 'MPD_L'])
n_mpd_L_Cl = len([mono for mono in mpds if mono.name == 'MPD_L_Cl'])
n_mpd_T = len([mono for mono in mpds if mono.name == 'MPD_T'])
n_tmc_mono = len([mono for mono in tmcs if mono.name == 'TMC'])
n_tmc_LD = len([mono for mono in tmcs if mono.name in ['TMC_L_D_1', 'TMC_L_D_2', 'TMC_L_D_3']])
n_tmc_LP = len([mono for mono in tmcs if mono.name in ['TMC_L_P_1', 'TMC_L_P_2', 'TMC_L_P_3']])
n_tmc_T0P = len([mono for mono in tmcs if mono.name in ['TMC_T_0P_1', 'TMC_T_0P_2', 'TMC_T_0P_3']])
n_tmc_T1P = len([mono for mono in tmcs if mono.name in ['TMC_T_1P_1_2', 'TMC_T_1P_1_3', 'TMC_T_1P_2_1', 'TMC_T_1P_2_3', 'TMC_T_1P_3_1', 'TMC_T_1P_3_2']])
n_tmc_T2P = len([mono for mono in tmcs if mono.name in ['TMC_T_2P_1', 'TMC_T_2P_2', 'TMC_T_2P_3']])
n_tmc_C = len([mono for mono in tmcs if mono.name == 'TMC_C'])

mpd_L = round(np.sum([charges_dict['MPD_L'][atom]['charge'] for atom in charges_dict['MPD_L']]),8)
mpd_L_Cl = round(np.sum([charges_dict['MPD_L_Cl'][atom]['charge'] for atom in charges_dict['MPD_L_Cl']]),8)
mpd_T = round(np.sum([charges_dict['MPD_T'][atom]['charge'] for atom in charges_dict['MPD_T']]),8)
tmc_LD = round(np.sum([charges_dict['TMC_L_D_1'][atom]['charge'] for atom in charges_dict['TMC_L_D_1']]),8)
tmc_LP = round(np.sum([charges_dict['TMC_L_P_1'][atom]['charge'] for atom in charges_dict['TMC_L_P_1']]),8)
tmc_T0P = round(np.sum([charges_dict['TMC_T_0P_1'][atom]['charge'] for atom in charges_dict['TMC_T_0P_1']]),8)
tmc_T1P = round(np.sum([charges_dict['TMC_T_1P_1_2'][atom]['charge'] for atom in charges_dict['TMC_T_1P_1_2']]),8)
tmc_T2P = round(np.sum([charges_dict['TMC_T_2P_1'][atom]['charge'] for atom in charges_dict['TMC_T_2P_1']]),8)
tmc_C = round(np.sum([charges_dict['TMC_C'][atom]['charge'] for atom in charges_dict['TMC_C']]),8)

print()
# print(f'MPD monomers: {n_mpd_mono}')
print(f'MPD-L: {n_mpd_L} with charge: {n_mpd_L*mpd_L:.4f}')
print(f'MPD-L-Cl: {n_mpd_L_Cl} with charge: {n_mpd_L_Cl*mpd_L_Cl:.4f}')
print(f'MPD-T: {n_mpd_T} with charge: {n_mpd_T*mpd_T:.4f}')
# print(f'TMC monomers: {n_tmc_mono}')
print(f'TMC-L-D: {n_tmc_LD} with charge: {n_tmc_LD*tmc_LD:.4f}')
print(f'TMC-L-P: {n_tmc_LP} with charge: {n_tmc_LP*tmc_LP:.4f}')
print(f'TMC-T-0P: {n_tmc_T0P} with charge: {n_tmc_T0P*tmc_T0P:.4f}')
print(f'TMC-T-1P: {n_tmc_T1P} with charge: {n_tmc_T1P*tmc_T1P:.4f}')
print(f'TMC-T-2P: {n_tmc_T2P} with charge: {n_tmc_T2P*tmc_T2P:.4f}')
print(f'TMC-C: {n_tmc_C} with charge: {n_tmc_C*tmc_C:.4f}')

n_deprot = n_tmc_LD + n_tmc_T0P*2 + n_tmc_T1P
print()
print(f'Total number of deprotonated groups: {n_deprot}')
print(f'Charge after reassignment: {atoms.charges.sum()}')


Creating MPD monomer fragments...


100%|██████████| 1432/1432 [00:07<00:00, 180.84it/s]


Creating TMC monomer fragments...


100%|██████████| 1464/1464 [00:08<00:00, 179.15it/s]


MPD-L: 97 with charge: -15.2096
MPD-L-Cl: 521 with charge: -103.7624
MPD-T: 98 with charge: -7.5421
TMC-L-D: 46 with charge: -34.4089
TMC-L-P: 46 with charge: 7.0803
TMC-T-0P: 6 with charge: -11.1283
TMC-T-1P: 7 with charge: -6.1384
TMC-T-2P: 6 with charge: 0.4704
TMC-C: 377 with charge: 73.6733

Total number of deprotonated groups: 65
Charge after reassignment: 5.909999956088541e-06
